# Exploration des données brutes — data/raw/events.json

Analyse exploratoire sur les 56 champs bruts, **avant** toute décision de filtrage ou de nettoyage. Objectif : établir les faits qui justifient ensuite les choix faits dans `preprocess_events.py` (exploration détaillée dans `02_preprocessing_walkthrough.ipynb`).

In [ ]:
import json
from collections import Counter
from pathlib import Path

DATA_PATH = Path("..") / "data" / "raw" / "events.json"
events = json.loads(DATA_PATH.read_text(encoding="utf-8"))

print("Nombre d'événements bruts :", len(events))
print("Nombre de champs par événement :", len(events[0].keys()))

## 1. Taux de valeurs manquantes sur les 56 champs bruts

In [ ]:
def is_missing(value) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    if isinstance(value, list):
        return len(value) == 0
    return False


total = len(events)
fields = list(events[0].keys())

print(f"{'Champ':<28}{'Manquants':>12}{'Taux':>10}")
for field in fields:
    missing_count = sum(1 for e in events if is_missing(e.get(field)))
    rate = missing_count / total * 100
    print(f"{field:<28}{missing_count:>12}{rate:>9.1f}%")

## 2. Classification des champs bruts (primaire / secondaire / inutile)

Avant d'aller plus loin, on classe les 56 champs bruts selon leur utilité pour ce chatbot :
- **`CHAMPS_PRIMAIRES`** : indispensables pour répondre aux questions de base (quoi / quand / où) — leur absence remettrait en cause l'utilité de l'événement lui-même
- **`CHAMPS_SECONDAIRES`** : enrichissent la réponse (tarifs, accessibilité, contact...) mais leur absence n'empêche pas de répondre à l'essentiel
- **`CHAMPS_INUTILES`** : aucune valeur exploitable pour ce chatbot (identifiants techniques, médias, données personnelles du contributeur, champs constants)

Chaque champ ci-dessous est affiché avec son taux de valeurs manquantes déjà calculé en section 1, pour confronter ce classement aux vrais chiffres plutôt qu'à une impression.

In [ ]:
CHAMPS_PRIMAIRES = [
    "uid", "title_fr", "description_fr", "longdescription_fr",
    "firstdate_begin", "lastdate_end",
    "location_name", "location_address", "location_city",
]

CHAMPS_SECONDAIRES = [
    "canonicalurl", "conditions_fr", "keywords_fr", "accessibility_label_fr",
    "attendancemode", "onlineaccesslink", "status", "age_min", "age_max", "registration",
    "location_district", "location_insee", "location_postalcode",
    "location_phone", "location_website", "location_links",
]

CHAMPS_INUTILES = [
    "slug", "image", "imagecredits", "thumbnail", "originalimage",
    "updatedat", "daterange_fr", "firstdate_end", "lastdate_begin", "timings",
    "accessibility", "location_uid", "location_coordinates",
    "location_department", "location_region", "location_countrycode",
    "location_image", "location_imagecredits", "location_tags",
    "location_description_fr", "location_access_fr",
    "originagenda_title", "originagenda_uid",
    "contributor_email", "contributor_contactnumber", "contributor_contactname",
    "contributor_contactposition", "contributor_organization",
    "category", "country_fr", "links",
]

categories = {
    "PRIMAIRES (critiques : quoi / quand / où)": CHAMPS_PRIMAIRES,
    "SECONDAIRES (enrichissent la réponse, non critiques)": CHAMPS_SECONDAIRES,
    "INUTILES (aucune valeur pour ce chatbot)": CHAMPS_INUTILES,
}

for label, champs in categories.items():
    print(f"=== {label} ({len(champs)} champs) ===")
    for field in champs:
        missing_count = sum(1 for e in events if is_missing(e.get(field)))
        rate = missing_count / total * 100
        print(f"  {field:<28}{missing_count:>8}{rate:>9.1f}%")
    print()

print("Total classé :", len(CHAMPS_PRIMAIRES) + len(CHAMPS_SECONDAIRES) + len(CHAMPS_INUTILES), "/", len(fields))

## 3. D'où viennent les événements ? (`originagenda_title`)

Ce champ dit quel organisme a publié l'événement. On y avait découvert qu'une seule source, France Travail, représente une part énorme du volume — à vérifier ici avec les vrais chiffres.

In [ ]:
origins = Counter(e.get("originagenda_title") for e in events)
for origin, count in origins.most_common(15):
    print(f"{count:>6}  {origin}")

In [ ]:
france_travail_count = origins["Mes événements France Travail"]
print(f"France Travail : {france_travail_count}/{total} ({france_travail_count/total*100:.1f}%)")

## 4. Mots-clés dominants (`keywords_fr`)

Confirme si le bruit repéré ("Recrutement", "S'informer"...) correspond bien à la source France Travail.

In [ ]:
keyword_counter = Counter()
for e in events:
    for kw in (e.get("keywords_fr") or []):
        keyword_counter[kw] += 1

for kw, count in keyword_counter.most_common(15):
    print(f"{count:>6}  {kw}")

## 5. Statut des événements (`status`)

Champ stocké en JSON imbriqué dans une chaîne de texte — on le décode ici pour voir la répartition réelle (Programmé / Annulé / Re-programmé).

In [ ]:
def parse_label(raw_value):
    if not raw_value:
        return None
    try:
        return json.loads(raw_value)["label"]["fr"]
    except (json.JSONDecodeError, KeyError, TypeError):
        return None


status_counter = Counter(parse_label(e.get("status")) for e in events)
for status, count in status_counter.most_common():
    print(f"{count:>6}  {status}")

## 6. Mode de participation (`attendancemode`)

In [ ]:
attendance_counter = Counter(parse_label(e.get("attendancemode")) for e in events)
for mode, count in attendance_counter.most_common():
    print(f"{count:>6}  {mode}")

## 7. Cohérence des noms de ville (`location_city` vs `location_insee`)

Un même code INSEE (= une même commune réelle) associé à plusieurs orthographes de ville différentes révèle soit une incohérence de casse, soit une vraie erreur de géocodage dans la donnée source.

In [ ]:
from collections import defaultdict

insee_to_city_names = defaultdict(set)
for e in events:
    insee = e.get("location_insee")
    city = e.get("location_city")
    if insee and city:
        insee_to_city_names[insee].add(city)

inconsistent = {insee: names for insee, names in insee_to_city_names.items() if len(names) > 1}
print(f"Codes INSEE avec plusieurs orthographes de ville : {len(inconsistent)}/{len(insee_to_city_names)}")
for insee, names in list(inconsistent.items())[:15]:
    print(insee, "->", names)

Le nombre de codes seul ne dit rien de l'ampleur réelle du problème — un code très utilisé (comme celui de Marseille) pèserait autant qu'un code rarement utilisé dans le décompte précédent. On compte ici directement le nombre d'**événements** concernés, pas de codes.

In [ ]:
affected_events = [e for e in events if e.get("location_insee") in inconsistent]

print(f"Codes INSEE incohérents : {len(inconsistent)} (sur {len(insee_to_city_names)} codes distincts)")
print(f"Événements réellement concernés par un de ces codes : {len(affected_events)}/{total} ({len(affected_events)/total*100:.1f}%)")

Le chiffre précédent mélange deux problèmes très différents : une simple variation de casse/tiret (déjà réglée par `normalize_city`, sans gravité) et une vraie divergence de commune (le seul cas qui pose un vrai problème). On sépare les deux ici, avec le détail événement par événement pour chaque code réellement incohérent.

In [ ]:
def normalize_for_comparison(name: str) -> str:
    return name.lower().replace("-", " ").strip()


inconsistent_case_only = {}
inconsistent_real = {}
for insee, names in inconsistent.items():
    normalized = {normalize_for_comparison(n) for n in names}
    if len(normalized) == 1:
        inconsistent_case_only[insee] = names
    else:
        inconsistent_real[insee] = names

case_only_events = [e for e in events if e.get("location_insee") in inconsistent_case_only]
real_events = [e for e in events if e.get("location_insee") in inconsistent_real]

print(f"Casse/tiret uniquement (déjà corrigé par normalize_city) : {len(inconsistent_case_only)} codes, "
      f"{len(case_only_events)} événements ({len(case_only_events)/total*100:.1f}%)")
print(f"Vraie incohérence (communes différentes)                : {len(inconsistent_real)} codes, "
      f"{len(real_events)} événements ({len(real_events)/total*100:.1f}%)")

print("\nDétail par variante, pour les codes réellement incohérents :")
for insee, names in inconsistent_real.items():
    print(f"\n{insee} :")
    for name in names:
        count = sum(1 for e in events if e.get("location_insee") == insee and e.get("location_city") == name)
        print(f"    {name:<30}{count:>6} événement(s)")

Pour décider si ces événements minoritaires doivent être supprimés ou non, il faut plus que le seul nom de ville en désaccord — l'adresse complète, le code postal et le département permettent de vérifier si l'événement est réellement hors zone (auquel cas le supprimer serait justifié) ou juste mal étiqueté sur `location_city`/`location_insee` alors qu'il est bien dans les Bouches-du-Rhône (auquel cas mieux vaut le garder).

In [ ]:
print("Détail complet des événements minoritaires (ceux qui contredisent la majorité pour leur code INSEE) :\n")

for insee, names in inconsistent_real.items():
    events_for_insee = [e for e in events if e.get("location_insee") == insee]
    majority_city = max(names, key=lambda n: sum(1 for e in events_for_insee if e.get("location_city") == n))
    minority_events = [e for e in events_for_insee if e.get("location_city") != majority_city]

    for e in minority_events:
        print(f"Titre               : {e.get('title_fr')}")
        print(f"location_city       : {e.get('location_city')}  (majorité pour ce code : {majority_city})")
        print(f"location_insee      : {e.get('location_insee')}")
        print(f"location_address    : {e.get('location_address')}")
        print(f"location_postalcode : {e.get('location_postalcode')}")
        print(f"location_department : {e.get('location_department')}")
        print("-" * 60)

## 8. Aperçu qualitatif : titres au hasard

In [ ]:
import random

random.seed(42)
for e in random.sample(events, 10):
    print("-", e.get("title_fr"))

## 9. Événements en présentiel sans localisation / en ligne sans lien d'accès

Deux cas symétriques à vérifier avant de coder une règle d'exclusion dans `preprocess_events.py` :
- Un événement en présentiel/mixte (`attendancemode` != "En ligne") sans aucune information de lieu (`location_name`, `location_address`, `location_city` tous vides) — personne ne peut savoir où se rendre.
- Un événement en ligne sans aucun lien pour y accéder (`onlineaccesslink` vide ET aucun lien d'inscription dans `registration`) — personne ne peut savoir comment y participer.

In [ ]:
def has_registration_link(event) -> bool:
    raw = event.get("registration")
    if not raw:
        return False
    try:
        entries = json.loads(raw)
        return bool(entries and entries[0].get("value"))
    except (json.JSONDecodeError, KeyError, TypeError, IndexError):
        return False


in_person_no_location = [
    e for e in events
    if parse_label(e.get("attendancemode")) != "En ligne"
    and not (e.get("location_name") or e.get("location_address") or e.get("location_city"))
]

online_no_link = [
    e for e in events
    if parse_label(e.get("attendancemode")) == "En ligne"
    and not e.get("onlineaccesslink")
    and not has_registration_link(e)
]

print(f"Présentiel/mixte sans aucune localisation : {len(in_person_no_location)}/{total} "
      f"({len(in_person_no_location)/total*100:.2f}%)")
print(f"En ligne sans lien d'accès ni inscription  : {len(online_no_link)}/{total} "
      f"({len(online_no_link)/total*100:.2f}%)")

print("\nExemples présentiel sans localisation :")
for e in in_person_no_location[:5]:
    print(" -", e.get("title_fr"))

print("\nExemples en ligne sans lien :")
for e in online_no_link[:5]:
    print(" -", e.get("title_fr"))

## Conclusion

Ces constats motivent directement les décisions prises dans `scripts/preprocess_events.py` :
- Volume France Travail élevé → exclusion de cette source (`is_relevant`)
- Événements "Annulé" présents → statut conservé et signalé plutôt que masqué
- Incohérences de casse sur `location_city` → normalisation (`normalize_city`)
- Champs `status`/`attendancemode` en JSON imbriqué → nécessité d'un parsing dédié (`parse_labeled_field`)

**Nouveau constat (section 7)** : 14 codes INSEE sur 103 sont associés à plusieurs noms de ville différents, mais l'analyse détaillée (casse/tiret vs vraie incohérence, puis vérification via le code postal de chaque événement minoritaire) montre que :
- L'écrasante majorité de ces "incohérences" ne sont pas des erreurs : simple casse (`MARSEILLE`/`Marseille`), suffixe postal (`Vitrolles cedex`), ou hameaux réels d'une même commune (Pont-de-Crau/Arles, Calas/Cabriès...)
- **3 événements sur 8298 (2x "LANDERNEAU", 1x "PIOLENC") ont un code postal qui ne correspond pas au département Bouches-du-Rhône** (29800 = Finistère, 84420 = Vaucluse), alors que `location_department` affirme à tort "Bouches-du-Rhône" — ces événements sont réellement hors zone
- Règle proposée pour `preprocess_events.py` : exclure les événements dont `location_postalcode` ne commence pas par `"13"`, plus fiable que `location_department` ou que la réconciliation `location_city`/`location_insee`

La vérification que ces décisions (existantes et nouvelle) sont bien appliquées, étape par étape, sur des exemples réels, se trouve dans `02_preprocessing_walkthrough.ipynb`.